# Sentinel-1 OCN — Sea Surface Wind and Radial Velocity
**Author:** Sadra Emamalizadeh

This notebook reads a Sentinel-1 IW OCN NetCDF file and plots:
- OWI wind speed (with direction arrows)
- OWI wind direction
- RVL radial surface velocity

**Data:** Sentinel-1 IW OCN product (`.SAFE` folder, `measurement/*.nc`)
Download from: [Copernicus Data Space](https://dataspace.copernicus.eu)


## 0. Imports

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

## 1. User Settings
**Change only this path** to point to your local OCN NetCDF file.

In [ ]:
# Path to the main OCN NetCDF file inside the .SAFE folder
# It is the file named: s1*-iw-ocn-vv-*.nc  (not the iw1/iw2/iw3 files)
nc_path = 'data/s1d-iw-ocn-vv-20260614t165724-20260614t165755-003235-005A58-001.nc'

## 2. Load Dataset

In [ ]:
ds = xr.open_dataset(nc_path, engine='h5netcdf')

# Extract the three variables we need
wind_speed = ds['owiWindSpeed']    # unit: m/s
wind_dir   = ds['owiWindDirection'] # unit: degrees (meteorological convention)
rvl        = ds['rvlRadVel']        # unit: m/s

print('Wind speed shape:', wind_speed.shape)
print('Wind dir shape:  ', wind_dir.shape)
print('Radvel shape:    ', rvl.shape)
print()
print('Wind speed range:', np.nanmin(wind_speed.values), 'to', np.nanmax(wind_speed.values), 'm/s')
print('Radvel range:    ', np.nanmin(rvl.values), 'to', np.nanmax(rvl.values), 'm/s')

## 3. Prepare Data

**Wind:** already 2D `(owiAzSize, owiRaSize)` — xarray merged the 3 swaths automatically on load.

**Radvel:** still has a swath dimension `(rvlAzSize, rvlRaSize, rvlSwath)` — we concatenate the 3 swaths along the range axis.

No fill values found in metadata, so we apply a safe large-number mask just in case.

In [ ]:
# Wind: use directly, mask any unrealistic large values
speed = wind_speed.where(wind_speed < 1e10)
direc = wind_dir.where(wind_dir < 1e10)

# Radvel: concatenate 3 swaths side by side along range dimension
radvel = xr.concat([rvl.isel(rvlSwath=i) for i in range(3)], dim='rvlRaSize')
radvel = radvel.where(radvel < 1e10)

print('After processing:')
print('Speed shape: ', speed.shape)
print('Radvel shape:', radvel.shape)

## 4. Compute Wind Vector Components for Arrows

Wind direction is given as degrees from North (meteorological convention).
We convert to east (u) and north (v) components so matplotlib can draw arrows:

- `u = speed × sin(direction)` → east-west component
- `v = speed × cos(direction)` → north-south component

`step=8` means we subsample every 8th pixel — otherwise 44,000 arrows would be unreadable.

In [ ]:
step = 8

u = speed.values[::step, ::step] * np.sin(np.deg2rad(direc.values[::step, ::step]))
v = speed.values[::step, ::step] * np.cos(np.deg2rad(direc.values[::step, ::step]))

## 5. Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Wind speed + direction arrows ──────────────────────────
im0 = axes[0].imshow(speed.values, cmap='Blues', vmin=0, vmax=15, origin='upper')
axes[0].quiver(
    np.arange(0, speed.shape[1], step),  # arrow x positions (columns)
    np.arange(0, speed.shape[0], step),  # arrow y positions (rows)
    u, v,
    color='black', scale=150, width=0.003
)
plt.colorbar(im0, ax=axes[0], label='Wind speed [m/s]')
axes[0].set_title('OWI Wind Speed + Direction')
axes[0].set_xlabel('Range pixels')
axes[0].set_ylabel('Azimuth pixels')

# ── Wind direction ─────────────────────────────────────────
im1 = axes[1].imshow(direc.values, cmap='hsv', vmin=0, vmax=360, origin='upper')
plt.colorbar(im1, ax=axes[1], label='Wind direction [°]')
axes[1].set_title('OWI Wind Direction')
axes[1].set_xlabel('Range pixels')

# ── Radial velocity ────────────────────────────────────────
# Diverging colormap: blue = moving toward satellite, red = moving away
im2 = axes[2].imshow(radvel.values, cmap='RdBu_r', vmin=-3, vmax=3, origin='upper')
plt.colorbar(im2, ax=axes[2], label='Radial velocity [m/s]')
axes[2].set_title('RVL Radial Velocity')
axes[2].set_xlabel('Range pixels')

plt.suptitle('Sentinel-1 OCN Products — IW Mode', fontsize=13)
plt.tight_layout()
plt.show()

# ── Quick statistics ───────────────────────────────────────
print(f'Wind speed:  {np.nanmean(speed.values):.1f} m/s mean, {np.nanmax(speed.values):.1f} m/s max')
print(f'Wind dir:    {np.nanmean(direc.values):.0f}° mean')
print(f'Radial vel:  {np.nanmean(radvel.values):.2f} m/s mean, '
      f'range {np.nanmin(radvel.values):.2f} to {np.nanmax(radvel.values):.2f} m/s')